# Test SNR-Aware Filtering Methods

**Purpose:** Demonstrate and compare different methods for handling poor SNR at wavelength edges.

**Problem:** After `apply_illumination_correction_v2()` and `apply_wavelength_filter()`, edge bands still have high noise due to low radiance values being amplified.

**Solution:** Apply soft weighting, SNR-based filtering, or adaptive smoothing to reduce edge band influence.

See `SPECTRAL_PREPROCESSING_WORKFLOW_AND_SNR_FILTERING.md` for full documentation.

## Setup

In [ ]:
import importlib
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../"))

from utils.gref_pipeline import georef
from gref_pipeline import config

importlib.reload(georef)
from utils.gref_pipeline.georef import *

print("✅ Module loaded successfully!")

## Load Data and Standard Preprocessing

In [ ]:
# Load transect 057_5 (known good data)
transect = load_transect(config.OUTPUT_FOLDER)
transect.list_files()

cube = transect.select_files(["rad_uhi_20241029_115057_5"])
cube.describe()

In [ ]:
# Apply illumination correction (converts to pseudo-reflectance)
cube.apply_illumination_correction_v2(window_size=500, strength=1.0)

print(f"✅ Illumination correction complete")
print(f"   Data shape: {cube.data_corrected.shape}")

In [ ]:
# Apply wavelength filter (crop to 490-690 nm)
cube.apply_wavelength_filter(wavelength_range=(490, 690))

print(f"✅ Wavelength filtering complete")
print(f"   Wavelength range: {cube.wavelengths[0]:.1f} - {cube.wavelengths[-1]:.1f} nm")
print(f"   Number of bands: {len(cube.wavelengths)}")

## Diagnose SNR Issues

Look at raw spectra to see edge band noise.

In [ ]:
# Plot some example spectra WITHOUT any filtering
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Different locations
locations = [(300, 500), (800, 500), (1200, 500), (1600, 500)]
colors = ["blue", "green", "red", "purple"]

for idx, ((t, s), color) in enumerate(zip(locations, colors)):
    ax = axes[idx // 2, idx % 2]
    spectrum = cube.data_corrected[t, s, :]

    ax.plot(cube.wavelengths, spectrum, color=color, linewidth=2, alpha=0.8)
    ax.set_xlabel("Wavelength (nm)", fontsize=11)
    ax.set_ylabel("Pseudo-reflectance", fontsize=11)
    ax.set_title(f"Raw Spectrum (track={t}, slit={s})", fontsize=12, fontweight="bold")
    ax.grid(True, alpha=0.3)

    # Highlight edge regions
    ax.axvspan(490, 510, alpha=0.2, color="red", label="Edge region")
    ax.axvspan(670, 690, alpha=0.2, color="red")
    ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig("test_raw_spectra_edge_noise.png", dpi=150, bbox_inches="tight")
plt.show()

print("📊 Notice the noise at edges (490-510 nm and 670-690 nm)")
print("   These bands have low radiance → illumination correction amplifies noise!")

## Method 1: Soft Edge Weighting

Apply sigmoid weights to gradually reduce edge band influence.

In [ ]:
def apply_soft_edge_weighting(datacube, wavelengths, edge_width=20, min_weight=0.1):
    """
    Apply sigmoid weights to reduce edge band influence.

    Parameters:
    -----------
    edge_width : float
        Distance (in nm) from each edge where weighting starts
    min_weight : float
        Minimum weight at the very edge (0-1)
    """
    wl_min, wl_max = wavelengths[0], wavelengths[-1]

    # Create sigmoid weights
    weights = np.ones(len(wavelengths))

    # Left edge
    left_mask = wavelengths < (wl_min + edge_width)
    if np.any(left_mask):
        x = (wavelengths[left_mask] - wl_min) / edge_width
        weights[left_mask] = min_weight + (1 - min_weight) * (
            1 / (1 + np.exp(-10 * (x - 0.5)))
        )

    # Right edge
    right_mask = wavelengths > (wl_max - edge_width)
    if np.any(right_mask):
        x = (wl_max - wavelengths[right_mask]) / edge_width
        weights[right_mask] = min_weight + (1 - min_weight) * (
            1 / (1 + np.exp(-10 * (x - 0.5)))
        )

    # Apply weights (broadcast across tracks and slits)
    weighted_cube = datacube * weights[np.newaxis, np.newaxis, :]

    return weighted_cube, weights

In [ ]:
# Save original data
data_original = cube.data_corrected.copy()

# Apply soft edge weighting
data_weighted, weights = apply_soft_edge_weighting(
    cube.data_corrected, cube.wavelengths, edge_width=20, min_weight=0.1
)

print("✅ Soft edge weighting applied")
print(f"   Edge width: 20 nm")
print(f"   Minimum weight: 0.1")
print(f"   Bands with weight < 1.0: {np.sum(weights < 1.0)}")

In [ ]:
# Plot weights
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(cube.wavelengths, weights, "b-", linewidth=3)
ax.set_xlabel("Wavelength (nm)", fontsize=12)
ax.set_ylabel("Weight", fontsize=12)
ax.set_title("Soft Edge Weighting Function", fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1.1])
ax.axhspan(0, 1, alpha=0.1, color="green")
plt.tight_layout()
plt.savefig("test_weight_function.png", dpi=150, bbox_inches="tight")
plt.show()

print("📊 Weight function: 1.0 in center, decays smoothly to 0.1 at edges")

In [ ]:
# Compare original vs weighted spectra
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for idx, ((t, s), color) in enumerate(zip(locations, colors)):
    ax = axes[idx // 2, idx % 2]

    spectrum_orig = data_original[t, s, :]
    spectrum_weighted = data_weighted[t, s, :]

    ax.plot(
        cube.wavelengths,
        spectrum_orig,
        color=color,
        linewidth=2,
        alpha=0.5,
        label="Original",
    )
    ax.plot(
        cube.wavelengths,
        spectrum_weighted,
        color="black",
        linewidth=2,
        alpha=0.8,
        label="Weighted",
    )

    ax.set_xlabel("Wavelength (nm)", fontsize=11)
    ax.set_ylabel("Pseudo-reflectance", fontsize=11)
    ax.set_title(
        f"Original vs Weighted (track={t}, slit={s})", fontsize=12, fontweight="bold"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig("test_original_vs_weighted.png", dpi=150, bbox_inches="tight")
plt.show()

print("📊 Black lines show reduced influence of edge bands")
print("   Center wavelengths (540-620 nm) unchanged")

## Method 2: SNR Calculation

Calculate Signal-to-Noise Ratio for each band to identify problematic wavelengths.

In [ ]:
def compute_snr_per_band(datacube, method="mad"):
    """
    Compute SNR for each spectral band.

    Parameters:
    -----------
    method : str
        'std': SNR = mean / std (simple)
        'mad': SNR based on Median Absolute Deviation (robust)
    """
    T, S, B = datacube.shape
    snr = np.zeros(B)

    for b in range(B):
        band_data = datacube[:, :, b].flatten()
        band_data = band_data[np.isfinite(band_data)]

        if method == "std":
            snr[b] = np.mean(band_data) / (np.std(band_data) + 1e-10)
        elif method == "mad":
            median = np.median(band_data)
            mad = np.median(np.abs(band_data - median))
            snr[b] = median / (1.4826 * mad + 1e-10)  # 1.4826 for normal distribution

    return snr

In [ ]:
# Compute SNR using MAD (robust method)
snr = compute_snr_per_band(data_original, method="mad")

print("✅ SNR computed for all bands")
print(f"\nSNR Statistics:")
print(f"  Min:    {np.min(snr):.2f}")
print(f"  25%:    {np.percentile(snr, 25):.2f}")
print(f"  Median: {np.median(snr):.2f}")
print(f"  75%:    {np.percentile(snr, 75):.2f}")
print(f"  Max:    {np.max(snr):.2f}")
print(f"\n  Bands with SNR < 10: {np.sum(snr < 10)}/{len(snr)}")
print(f"  Bands with SNR < 5:  {np.sum(snr < 5)}/{len(snr)}")

In [ ]:
# Plot SNR
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# SNR vs wavelength
ax1.plot(cube.wavelengths, snr, "b-", linewidth=2)
ax1.set_xlabel("Wavelength (nm)", fontsize=12)
ax1.set_ylabel("SNR (MAD method)", fontsize=12)
ax1.set_title("Signal-to-Noise Ratio per Band", fontsize=14, fontweight="bold")
ax1.grid(True, alpha=0.3)
ax1.axhline(y=10, color="r", linestyle="--", linewidth=2, label="Threshold=10")
ax1.axhline(y=5, color="orange", linestyle="--", linewidth=2, label="Threshold=5")
ax1.legend()

# Highlight low SNR regions
low_snr_mask = snr < 10
if np.any(low_snr_mask):
    for i, is_low in enumerate(low_snr_mask):
        if is_low:
            ax1.axvspan(
                cube.wavelengths[i] - 1, cube.wavelengths[i] + 1, alpha=0.1, color="red"
            )

# SNR histogram
ax2.hist(snr, bins=50, edgecolor="black", alpha=0.7, color="steelblue")
ax2.set_xlabel("SNR", fontsize=12)
ax2.set_ylabel("Count", fontsize=12)
ax2.set_title("SNR Distribution", fontsize=14, fontweight="bold")
ax2.axvline(x=10, color="r", linestyle="--", linewidth=2, label="Threshold=10")
ax2.axvline(x=5, color="orange", linestyle="--", linewidth=2, label="Threshold=5")
ax2.legend()

plt.tight_layout()
plt.savefig("test_snr_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n📊 SNR plot shows which bands have poor quality")
print("   Red shaded regions: SNR < 10 (problematic)")

## Method 3: Compare Different Edge Widths

Test different parameters to find optimal settings.

In [ ]:
# Test different edge widths
edge_widths = [10, 20, 30, 40]
min_weight = 0.1

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Pick one representative spectrum
t, s = 1200, 500
spectrum_orig = data_original[t, s, :]

for idx, edge_width in enumerate(edge_widths):
    ax = axes[idx // 2, idx % 2]

    # Apply weighting
    data_test, weights_test = apply_soft_edge_weighting(
        data_original, cube.wavelengths, edge_width=edge_width, min_weight=min_weight
    )
    spectrum_weighted = data_test[t, s, :]

    # Plot
    ax.plot(
        cube.wavelengths, spectrum_orig, "b-", linewidth=2, alpha=0.4, label="Original"
    )
    ax.plot(
        cube.wavelengths,
        spectrum_weighted,
        "r-",
        linewidth=2,
        alpha=0.8,
        label="Weighted",
    )
    ax.plot(
        cube.wavelengths, weights_test, "g--", linewidth=2, alpha=0.6, label="Weights"
    )

    ax.set_xlabel("Wavelength (nm)", fontsize=11)
    ax.set_ylabel("Value", fontsize=11)
    ax.set_title(f"Edge Width = {edge_width} nm", fontsize=12, fontweight="bold")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig("test_different_edge_widths.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n📊 Compare different edge width settings")
print("   edge_width=20 nm is recommended for most cases")
print("   Larger values = more gradual transition")

## Summary & Recommendations

### Key Findings:

1. **Edge bands (490-510 nm, 670-690 nm) have significantly lower SNR**
2. **Illumination correction amplifies noise at edges** (division by low values)
3. **Soft edge weighting effectively reduces edge noise influence**

### Recommended Workflow:

```python
# Standard preprocessing
cube.apply_illumination_correction_v2(window_size=500)
cube.apply_wavelength_filter(wavelength_range=(490, 690))

# NEW: Apply soft edge weighting
cube.data_corrected, weights = apply_soft_edge_weighting(
    cube.data_corrected,
    cube.wavelengths,
    edge_width=20,
    min_weight=0.1
)

# Continue with analysis
cube.apply_spectral_normalization(method='l2')
# ...
```

### Parameters:
- **edge_width=20**: Good balance between smoothness and preservation
- **min_weight=0.1**: Retains 10% of edge band information (not completely removed)

### Next Steps:
1. Test on SVM classification (compare accuracy with/without weighting)
2. Test on NDI analysis (check if edge bands affect results)
3. Consider adding to standard preprocessing pipeline
4. Document optimal parameters for different transects

In [ ]:
print("\n" + "=" * 80)
print("✅ TEST COMPLETE")
print("=" * 80)
print("\nGenerated files:")
print("  - test_raw_spectra_edge_noise.png")
print("  - test_weight_function.png")
print("  - test_original_vs_weighted.png")
print("  - test_snr_analysis.png")
print("  - test_different_edge_widths.png")
print("\nRecommendation: Use edge_width=20, min_weight=0.1")
print(
    "\nSee SPECTRAL_PREPROCESSING_WORKFLOW_AND_SNR_FILTERING.md for full documentation."
)